# LAB 2 - VMD and the PDB Format
**Looking at CLN025**


Authors:

- Prof. Marco A. Deriu (marco.deriu@polito.it)
- Eric A. Zizzi (eric.zizzi@polito.it)
- Marcello Miceli (marcello.miceli@polito.it)

# Table of Contents

1. The PDB format
2. Numbers from structures: text data files
3. VMD
4. Your deliverable

**Learning outcomes:**
- read a PDB file and know what each column means
- load and plot data from text files with Python
- display a protein in VMD with different representations, colours and selections
- measure distances and render a publication-quality image

# Why look at a structure?

Before computing anything, look at your system. Is there something wrong with this one?

<img src="broken-structure-example.png" width="900">

Many problems (a broken chain, a bond stretched across the box, a molecule outside the water) are obvious at a glance and invisible in a table of numbers.

# 1. The PDB format

The course molecule, CLN025, is the PDB entry [2RVD](https://www.rcsb.org/structure/2RVD), solved by NMR. A copy is in `../common/structures/2RVD.pdb`. A PDB file is plain text: every line is a **record** whose type is written in its first six characters.

| Record | Content |
|---|---|
| `HEADER`, `TITLE`, `COMPND`, `SOURCE` | What the entry is |
| `REMARK` | Free-text notes: experiment, refinement, missing atoms |
| `SEQRES` | The full sequence, including residues missing from the coordinates |
| `CRYST1` | Unit cell (for NMR entries a dummy 1 Å cell) |
| `MODEL` / `ENDMDL` | Start and end of one model; NMR entries have several |
| `ATOM` | Coordinates of an atom of a standard residue |
| `HETATM` | Coordinates of any other atom: ligands, ions, water, caps |
| `TER` | End of a chain |
| `END` | End of the file |

In [ ]:
!head -n 5 ../common/structures/2RVD.pdb
!grep "^SEQRES" ../common/structures/2RVD.pdb
!grep "^CRYST1" ../common/structures/2RVD.pdb
!echo "models: $(grep -c '^MODEL' ../common/structures/2RVD.pdb)"

## Fixed columns

`ATOM` and `HETATM` lines have **fixed columns**: every field sits at the same character positions, whatever its content.

| Columns | Field | Example |
|---|---|---|
| 1–6 | Record name | `ATOM` |
| 7–11 | Atom serial number | `2` |
| 13–16 | Atom name | `CA` |
| 17 | Alternate location (altloc) | blank, or `A`/`B` for atoms seen in two positions |
| 18–20 | Residue name | `TYR` |
| 22 | Chain | `A` |
| 23–26 | Residue number | `1` |
| 31–38, 39–46, 47–54 | x, y, z in Å | `2.093` |
| 55–60 | Occupancy | `1.00` (fraction of the time the atom is in this position) |
| 61–66 | B-factor | how much the atom moves or how uncertain it is; meaningful in X-ray structures |
| 77–78 | Element | `C` |

Splitting a line at spaces works most of the time, but breaks when two fields touch (a coordinate of −100.000 Å, a four-letter atom name next to the residue name). Reading by column always works:

In [ ]:
# first atoms of model 1, read by column
with open("../common/structures/2RVD.pdb") as pdb:
    atoms = [line for line in pdb if line.startswith("ATOM")][:12]

print(f"{'serial':>6} {'name':<4} {'res':<3} {'chain':<5} {'resid':>5} {'x':>8} {'y':>8} {'z':>8} {'occ':>5} {'B':>6}")
for line in atoms:
    print(f"{line[6:11].strip():>6} {line[12:16].strip():<4} {line[17:20]:<3} {line[21]:<5} {line[22:26].strip():>5} "
          f"{float(line[30:38]):8.3f} {float(line[38:46]):8.3f} {float(line[46:54]):8.3f} {line[54:60].strip():>5} {line[60:66].strip():>6}")

## ATOM or HETATM?

From LAB 3 on you simulate CLN025 with both ends capped: an acetyl group (ACE) on Tyr1 and an amide group (NH2) on Tyr10. The caps are not standard amino acids, so they are written as `HETATM` records:

In [ ]:
!grep -E "^(HETATM|TER|END)" ../common/structures/cln025_capped.pdb

# 2. Numbers from structures: text data files

The 20 NMR models of 2RVD are not identical: they show how much the structure varies. Measure one number on each model, the distance between the Cα atoms of Tyr1 and Tyr10 (the **end-to-end distance** of the hairpin), reading the file by column as above. `MODEL` lines tell which model the next atoms belong to:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ends = {}      # (model, residue number) -> coordinates of the CA atom, in Å
with open("../common/structures/2RVD.pdb") as pdb:
    for line in pdb:
        if line.startswith("MODEL"):
            model = int(line.split()[1])
        elif line.startswith("ATOM") and line[12:16].strip() == "CA" and int(line[22:26]) in (1, 10):
            ends[(model, int(line[22:26]))] = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])

models = sorted({m for m, residue in ends})
distance = [np.linalg.norm(ends[(m, 10)] - ends[(m, 1)]) for m in models]   # length of the vector Tyr1 CA -> Tyr10 CA
print(f"{len(models)} models, model 1: {distance[0]:.2f} Å")

Save the numbers in a **text data file**: columns of numbers, one line per model, under a few comment lines that say what the columns are. Almost every scientific program shares its results this way, GROMACS included (LAB 3).

In [ ]:
import os

os.makedirs("analysis", exist_ok=True)
with open("analysis/e2e.dat", "w") as out:
    out.write("# End-to-end distance of the 20 NMR models of PDB 2RVD\n")
    out.write("# column 1: model, column 2: Tyr1 CA - Tyr10 CA distance (Å)\n")
    for m, d in zip(models, distance):
        out.write(f"{m:4d} {d:8.3f}\n")

In [ ]:
!head -n 5 analysis/e2e.dat

`numpy.loadtxt` reads such a file back into an array, one row per line and one column per column, skipping the lines that start with the comment character (`#` unless told otherwise):

In [ ]:
data = np.loadtxt("analysis/e2e.dat", comments="#")
model, distance = data[:, 0].astype(int), data[:, 1]

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 4))
left.plot(model, distance, "o-")
left.set_xlabel("NMR model")
left.set_ylabel("Tyr1 CA - Tyr10 CA distance (Å)")
left.set_xticks(model[::2])
right.hist(distance, bins=8, edgecolor="black")
right.set_xlabel("Tyr1 CA - Tyr10 CA distance (Å)")
right.set_ylabel("Number of models")
plt.tight_layout()

print(f"mean {distance.mean():.2f} Å, standard deviation {distance.std(ddof=1):.2f} Å, "
      f"range {distance.min():.2f}-{distance.max():.2f} Å (most extended: model {model[distance.argmax()]})")

# 3. VMD

[VMD](https://www.ks.uiuc.edu/Research/vmd/) (Visual Molecular Dynamics) displays and analyses structures and trajectories. Start it from the **Applications** menu, or from a terminal in this folder:

```bash
vmd ../common/structures/2RVD.pdb
```

VMD opens several windows:

| Window | Use |
|---|---|
| **VMD Main** | Loaded molecules, frame slider, menus |
| **OpenGL Display** | The 3D view: left-drag rotates, right-drag (or Ctrl+drag) moves, scroll zooms |
| **Graphics → Representations** | How each part of the molecule is drawn |
| **Extensions → Tk Console** | A command line for VMD's scripting language, Tcl |

`vmd-tutorial.pdf` in this folder is the official VMD tutorial if you want to go further.

## 3.1 Representations

A representation is **what** is drawn (a selection), **how** (drawing method) and in **which colours** (colouring method). Open **Graphics → Representations** and change the drawing method of the default representation:

| Drawing method | Shows | Good for |
|---|---|---|
| Lines | Bonds as thin lines | Quick look, big systems |
| CPK | Balls and sticks | Atoms and bonds together |
| Licorice | Sticks | Side chains, ligands |
| VDW | Spheres at the van der Waals radius | Packing, contacts |
| NewCartoon | Ribbon following the backbone | Secondary structure |
| QuickSurf | Smooth molecular surface | Shape, pockets |

NewCartoon needs to know the secondary structure of every residue. VMD computes it with the program STRIDE from the coordinates; Lines, CPK and the others do not need it.

**Try it:** draw the whole molecule as NewCartoon, then click **Create Rep** and draw Tyr2 and Trp9 as Licorice (selection `resid 2 9 and sidechain`).

## 3.2 Colouring methods

| Colouring method | Colour by |
|---|---|
| Name | Element of the atom (C cyan, N blue, O red, H white) |
| ResName | Residue name |
| ResType | Residue type: polar, nonpolar, acidic, basic |
| Structure | Secondary structure (β-strand yellow, turn cyan, coil white) |
| Beta | Value in the B-factor column (with the Color Scale menu) |
| Index | Atom index, as a gradient |
| ColorID | One fixed colour you choose |

**Try it:** colour the cartoon by Structure. Where are the two β-strands, and where is the turn?

## 3.3 The selection language

Every representation, measurement and script uses the same selection language:

| Selection | Atoms selected |
|---|---|
| `name CA` | All Cα atoms |
| `resid 2 9` | Residues 2 and 9 |
| `resname TYR` | All tyrosines |
| `protein and noh` | Protein heavy atoms (no hydrogens) |
| `backbone` / `sidechain` | Backbone or side-chain atoms |
| `within 3.5 of resname TRP` | Atoms closer than 3.5 Å to any tryptophan atom |
| `same residue as (within 4 of resid 9)` | Whole residues with any atom near residue 9 |

Combine them with `and`, `or`, `not` and parentheses.

**Try it:** which residues touch Trp9? Draw `same residue as (within 4 of resid 9 and sidechain)` as Licorice.

## 3.4 One molecule, 20 models

2RVD contains 20 models, which VMD loads as 20 frames. Move the frame slider of the **VMD Main** window, or press **Play**, to go through them. To see them all at once: in **Graphics → Representations**, open the **Trajectory** tab and set **Draw Multiple Frames** to `0:19`.

Which parts of the hairpin vary most between models?

## 3.5 The same from the Tk Console

Everything above can be typed in **Extensions → Tk Console**. Scripts make a view easy to reproduce:

```tcl
mol new ../common/structures/2RVD.pdb waitfor all
mol delrep 0 top
mol representation NewCartoon
mol color Structure
mol selection all
mol addrep top
mol representation Licorice 0.3 12 12
mol color ResName
mol selection "resid 2 9 and sidechain"
mol addrep top
```

## 3.6 Measurements

Press **2** in the OpenGL Display and click two atoms: VMD draws the distance between them. **3** measures an angle (three atoms), **4** a dihedral (four atoms), and **1** goes back to rotating. **Graphics → Labels** lists every measurement, and its **Graph** tab plots it over all frames.

The hairpin is held together by backbone hydrogen bonds across the turn and by the packing of Tyr2 against Trp9. Measure in model 1:

| Pair | What it is |
|---|---|
| Asp3 N – Thr8 O | Backbone hydrogen bond |
| Gly7 N – Asp3 O | Backbone hydrogen bond |
| Thr8 N – Asp3 O | Backbone hydrogen bond |
| Tyr2 Cα – Trp9 Cα | Across the hydrophobic core |
| Tyr1 Cα – Tyr10 Cα | End-to-end distance |

A hydrogen bond has its donor N and acceptor O about 2.8–3.5 Å apart.

The next cell runs VMD without a window and measures all five distances, plus the distance between the aromatic rings of Tyr2 and Trp9, in every model. Compare model 1 with your own measurements.

In [ ]:
%%bash
cat > analysis/measure.tcl <<'EOF'
mol new ../common/structures/2RVD.pdb waitfor all
set n [molinfo top get numframes]
proc report {label values} {
    puts [format "RESULT %-22s model 1 %5.2f   min %5.2f   max %5.2f   (A)" $label \
        [lindex $values 0] [tcl::mathfunc::min {*}$values] [tcl::mathfunc::max {*}$values]]
}
foreach {label s1 s2} {
    "Asp3 N - Thr8 O"      "resid 3 and name N"  "resid 8 and name O"
    "Gly7 N - Asp3 O"      "resid 7 and name N"  "resid 3 and name O"
    "Thr8 N - Asp3 O"      "resid 8 and name N"  "resid 3 and name O"
    "Tyr2 CA - Trp9 CA"    "resid 2 and name CA" "resid 9 and name CA"
    "Tyr1 CA - Tyr10 CA"   "resid 1 and name CA" "resid 10 and name CA"
} {
    set a [atomselect top $s1]
    set b [atomselect top $s2]
    report $label [measure bond [list [$a get index] [$b get index]] frame all]
}
# centre of the six-membered rings of Tyr2 and Trp9
set r1 [atomselect top "resid 2 and name CG CD1 CD2 CE1 CE2 CZ"]
set r2 [atomselect top "resid 9 and name CD2 CE2 CE3 CZ2 CZ3 CH2"]
set rings {}
for {set f 0} {$f < $n} {incr f} {
    $r1 frame $f
    $r2 frame $f
    lappend rings [vecdist [measure center $r1] [measure center $r2]]
}
report "Tyr2 ring - Trp9 ring" $rings
quit
EOF
vmd -dispdev text -e analysis/measure.tcl 2>&1 | grep "^RESULT"

## 3.7 Rendering

The OpenGL Display is for working; for a figure, render the scene with a ray tracer:

1. **Graphics → Colors → Display → Background**: white.
2. **Display**: switch **Depth Cueing** off (or tune it in **Display Settings**) and choose **Orthographic** projection.
3. **Graphics → Materials**: try `AOChalky` or `Glossy` for the cartoon.
4. Make the OpenGL Display window as large as the image you want: the render has the same size in pixels.
5. **File → Render**: choose **Tachyon (internal, in-memory rendering)**, set the file name to `cln025.tga` and click **Start Rendering**.
6. **File → Save Visualization State** saves the whole view as a `.vmd` file; `vmd -e cln025.vmd` restores it later.

Tachyon writes TGA images. Convert yours to PNG for slides and reports:

In [ ]:
from PIL import Image

Image.open("cln025.tga").save("cln025.png")

# 4. Your deliverable

Keep both of these: you will reuse them for the rest of the course.

1. **A publication-quality render** of the CLN025 hairpin (model 1): cartoon coloured by secondary structure, Tyr2 and Trp9 (the hydrophobic core) as Licorice, white background, saved as PNG together with its `.vmd` state file.
2. **A table of the distances** of section 3.6: your own measurements in model 1 and the range over the 20 models from the script. Which interactions vary most between models?